# XFeat 온도계 ABS: CPU 하드웨어 가속 실험 (2026-10-06)
새 파일이며 기존 실험 파일은 수정하지 않습니다. 고정 분위수 8-bin, 64차원, 7×uint64 코드를 유지합니다.

시도: C++ POPCNT 거리 / 양방향 MNN 통합 / OpenMP 1·2 threads / AVX-512 VPOPCNT 지원 시 실행 / 정확한 부분합 pruning / Faiss IndexBinaryFlat.
추가로 CPU 양자화와 packing을 C++ 한 함수로 합칩니다. SIMD는 명령 지원을 확인한 뒤에만 실행합니다.

**Run all**: 기존 COCO/HPatches Drive ZIP 캐시를 재사용합니다. TOP_K 기본값은 이전 온도계 기준선과 같은 800이며 1000으로 바꿀 수 있습니다. 다른 TOP_K 실험과 수치를 직접 비교하지 마세요.
모든 exact 방법은 ABS의 거리와 첫 인덱스 동점 규칙에 대해 검증합니다. Faiss는 range_search로 동점을 보정하며 그 비용도 시간에 포함합니다.
최종 SPEED 표는 cached·인코딩 포함 total·원본 float 대비 배수를 표시합니다. XFeat 추출, 경계 calibration, 컴파일, 정확도 검증은 매칭 시간에서 제외합니다.
GPU 및 근사 검색은 이 파일의 실험 범위에 포함하지 않습니다. 아직 실제 이미지 실행 결과를 채워 넣지 않았습니다.

오픈소스: [Faiss](https://github.com/facebookresearch/faiss/wiki/Binary-Indexes). 자체 C++ 구현은 CPU intrinsic과 OpenMP를 사용합니다.


In [1]:
!pip -q install opencv-python-headless numba faiss-cpu pandas

import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from numba import njit, prange, set_num_threads, get_num_threads
from google.colab import files, drive

SEED = 7
TOP_K = 800
LEVELS = 8  # seven thresholds per descriptor dimension
CALIBRATION_WARPS_PER_IMAGE = 6
VALIDATION_WARPS_PER_IMAGE = 3
MAX_POSITIVES_PER_PAIR = 500
GEOMETRIC_THRESHOLD = 3.0
NUMBA_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 3
REPEATS = 12

set_num_threads(NUMBA_THREADS)
torch.set_num_threads(NUMBA_THREADS)
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K).to(device).eval()
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
# COCO is used for calibration and an internal synthetic-warp check.
# No projection or threshold learning. HPatches is reserved for the external real-photo evaluation at the end.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'https://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    calibration_images = validation_images = images
    print('WARNING: One source image only; validation shares scene content with training.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.2 MB/s eta 0:00:00
The repository verlab_accelerated_features does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/verlab/accelerated_features/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/verlab/accelerated_features/raw/main/weights/xfeat.pt" to /root/.cache/torch/hub/checkpoints/xfeat.pt


100%|██████████| 5.96M/5.96M [00:00<00:00, 26.6MB/s]


Device: cpu
Mounted at /content/drive
Persistent dataset cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets
Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/val2017.zip
Selected 24 COCO images using seed 7; no COCO annotations needed.
Calibration files: ['000000002261.jpg', '000000029675.jpg', '000000067534.jpg', '000000162858.jpg', '000000165713.jpg', '000000173008.jpg', '000000175364.jpg', '000000194506.jpg', '000000270402.jpg', '000000289343.jpg', '000000360097.jpg', '000000460841.jpg', '000000471869.jpg', '000000479596.jpg', '000000504711.jpg', '000000517832.jpg', '000000528524.jpg', '000000547383.jpg']
Held-out files: ['000000077595.jpg', '000000132544.jpg', '000000334006.jpg', '000000394559.jpg', '000000449432.jpg', '000000474039.jpg']
Calibration images: 18 Validation images: 6


## 1. 특징 추출과 calibration / 검증 분리

XFeat는 고정합니다. 기존 프로토콜과 같은 calibration 이미지의 합성 warp descriptor에서 분위수 경계를 계산합니다. 대응점 계산은 기존 샘플링 순서를 유지하기 위한 것이며, 가중치를 학습하지 않습니다.


In [2]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.inference_mode()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': F.normalize(output['descriptors'].detach().cpu().float(), dim=-1),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def ground_truth_pairs(keypoints_1, keypoints_2, homography):
    if not len(keypoints_1) or not len(keypoints_2):
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    distances_12 = torch.cdist(project_points(keypoints_1, homography), keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors = distances_12[torch.arange(len(keypoints_1)), nearest_12]
    distances_21 = torch.cdist(project_points(keypoints_2, np.linalg.inv(homography)), keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)
    indices = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices) & (errors < GEOMETRIC_THRESHOLD)
    return indices[good], nearest_12[good]

def build_pairs(source_images, warps_per_image, generator):
    pairs = []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(warps_per_image):
            h = make_random_homography(*image.shape[:2], generator)
            warped = extract_features(warp_image(image, h, generator))
            i, j = ground_truth_pairs(base['keypoints'], warped['keypoints'], h)
            if len(i) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(i))[:MAX_POSITIVES_PER_PAIR]
                i, j = i[keep], j[keep]
            if len(i) < 32:
                print(f'Warning: image {image_index}, warp {warp_index}: {len(i)} positive pairs')
            pairs.append({'d1': base['descriptors'], 'd2': warped['descriptors'],
                          'k1': base['keypoints'], 'k2': warped['keypoints'],
                          'pos1': i, 'pos2': j, 'H': h})
    return pairs

start = time.perf_counter()
calibration_pairs = build_pairs(calibration_images, CALIBRATION_WARPS_PER_IMAGE, rng)
validation_pairs = build_pairs(validation_images, VALIDATION_WARPS_PER_IMAGE, rng)
calibration = torch.cat([torch.cat([p['d1'], p['d2']], dim=0) for p in calibration_pairs], dim=0)
if len(calibration) > 50000:
    calibration = calibration[torch.randperm(len(calibration))[:50000]]
print('Feature extraction time (calibration and validation):', round(time.perf_counter() - start, 2), 's')
print('Calibration pairs:', len(calibration_pairs), 'Validation pairs:', len(validation_pairs),
      'Calibration descriptors:', len(calibration))

Feature extraction time (calibration and validation): 23.61 s
Calibration pairs: 108 Validation pairs: 18 Calibration descriptors: 50000


In [3]:
# Fixed training-partition quantile edges; no optimizer, projection, or threshold learning.
base_edges_cpu = torch.quantile(calibration, torch.arange(1, LEVELS).float() / LEVELS, dim=0).T.contiguous()

def hard_quantize(descriptors, edges):
    return (descriptors[:, :, None] > edges[None, :, :]).sum(dim=-1).to(torch.uint8)

print('Fixed edges:', tuple(base_edges_cpu.shape), '| learned parameters: 0')


Fixed edges: (64, 7) | learned parameters: 0


## 2. 검증 기준 커널과 packed Thermometer

ABS와 Thermometer는 같은 8구간 L1 거리입니다. 모든 거리행렬과 MNN 인덱스를 검사합니다. popcount는 LLVM ctpop을 사용합니다. 이 셀에서는 pruning을 하지 않습니다.


In [4]:
@njit(parallel=True, cache=False)
def pairwise_uint8_abs(code_1, code_2):
    out = np.empty((code_1.shape[0], code_2.shape[0]), np.uint16)
    for i in prange(code_1.shape[0]):
        for j in range(code_2.shape[0]):
            total = 0
            for k in range(code_1.shape[1]):
                a, b = int(code_1[i, k]), int(code_2[j, k])
                total += a - b if a >= b else b - a
            out[i, j] = total
    return out

def mutual_from_scores(scores, higher_is_better):
    if not scores.shape[0] or not scores.shape[1]:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    if higher_is_better:
        nearest_12, nearest_21 = scores.argmax(axis=1), scores.argmax(axis=0)
    else:
        nearest_12, nearest_21 = scores.argmin(axis=1), scores.argmin(axis=0)
    indices = np.arange(scores.shape[0])
    keep = nearest_21[nearest_12] == indices
    return indices[keep], nearest_12[keep]

def encode(desc, edges):
    with torch.no_grad():
        levels = hard_quantize(desc.to(device), edges.to(device))
    return np.ascontiguousarray(levels.cpu().numpy(), dtype=np.uint8)

def match_float(pair):
    scores = (pair['d1'] @ pair['d2'].T).numpy()
    return mutual_from_scores(scores, higher_is_better=True)

def match_abs(pair, edges):
    a, b = encode(pair['d1'], edges), encode(pair['d2'], edges)
    return mutual_from_scores(pairwise_uint8_abs(a, b), higher_is_better=False)


# Packed thermometer: same 8-bin L1 distance, different computation.
from numba import types
from numba.extending import intrinsic
from llvmlite import ir

@intrinsic
def thermometer_popcount64(typing_context, value_type):
    if value_type != types.uint64:
        raise TypeError('thermometer_popcount64 expects uint64')
    signature = types.uint64(types.uint64)
    def codegen(context, builder, signature, arguments):
        function = builder.module.declare_intrinsic('llvm.ctpop', [ir.IntType(64)])
        return builder.call(function, arguments)
    return signature, codegen

@njit(cache=False)
def _pack_thermometer64(codes):
    packed = np.zeros((codes.shape[0], 7), dtype=np.uint64)
    for row in range(codes.shape[0]):
        for threshold in range(1, 8):
            word = np.uint64(0)
            for dimension in range(64):
                if codes[row, dimension] >= threshold:
                    word |= np.uint64(1) << np.uint64(dimension)
            packed[row, threshold - 1] = word
    return packed

def pack_thermometer64(codes):
    codes = np.asarray(codes)
    if codes.ndim != 2 or codes.shape[1] != 64:
        raise ValueError('Expected (N, 64) bin indices')
    if not np.issubdtype(codes.dtype, np.integer):
        raise ValueError('Bin indices must be integers')
    if codes.size and (codes.min() < 0 or codes.max() > 7):
        raise ValueError('Expected bin indices 0..7')
    return _pack_thermometer64(np.ascontiguousarray(codes, dtype=np.uint8))

@njit(parallel=True, cache=False)
def pairwise_thermometer64(packed_1, packed_2):
    distances = np.empty((packed_1.shape[0], packed_2.shape[0]), dtype=np.uint16)
    for i in prange(packed_1.shape[0]):
        for j in range(packed_2.shape[0]):
            total = np.uint64(0)
            for plane in range(7):
                total += thermometer_popcount64(packed_1[i, plane] ^ packed_2[j, plane])
            distances[i, j] = total
    return distances

def match_thermometer(pair, edges):
    a = pack_thermometer64(encode(pair['d1'], edges))
    b = pack_thermometer64(encode(pair['d2'], edges))
    return mutual_from_scores(pairwise_thermometer64(a, b), higher_is_better=False)

def verify_thermometer_codes(a, b):
    abs_distances = pairwise_uint8_abs(a, b)
    thermo_distances = pairwise_thermometer64(pack_thermometer64(a), pack_thermometer64(b))
    np.testing.assert_array_equal(abs_distances, thermo_distances)
    if len(a) and len(b):
        abs_matches = mutual_from_scores(abs_distances, False)
        thermo_matches = mutual_from_scores(thermo_distances, False)
        for abs_indices, thermo_indices in zip(abs_matches, thermo_matches):
            np.testing.assert_array_equal(abs_indices, thermo_indices)
    return True

if LEVELS != 8:
    raise ValueError('This experiment is for 8 bins and 64 dimensions')
# Compile before measurements; test all 8x8 scalar values at bits 0 and 63.
for dimension in (0, 63):
    scalar_codes = np.zeros((8, 64), dtype=np.uint8)
    scalar_codes[:, dimension] = np.arange(8, dtype=np.uint8)
    verify_thermometer_codes(scalar_codes, scalar_codes)
    expected = np.abs(np.arange(8)[:, None] - np.arange(8)[None, :])
    np.testing.assert_array_equal(pairwise_thermometer64(
        pack_thermometer64(scalar_codes), pack_thermometer64(scalar_codes)), expected)
test_rng = np.random.default_rng(SEED)
verify_thermometer_codes(test_rng.integers(0, 8, (19, 64), dtype=np.uint8),
                         test_rng.integers(0, 8, (23, 64), dtype=np.uint8))
zeros = np.zeros((1, 64), dtype=np.uint8)
sevens = np.full((1, 64), 7, dtype=np.uint8)
verify_thermometer_codes(zeros, sevens)
assert int(pairwise_thermometer64(pack_thermometer64(zeros), pack_thermometer64(sevens))[0, 0]) == 448
assert pack_thermometer64(zeros).nbytes == 56
print('PASS: scalar cases, highest bit, random distances/MNN, maximum distance 448.')
print('Storage per descriptor: float32 256 B | uint8 ABS 64 B | packed thermometer 56 B.')
print('Thermometer is 7 threshold planes, not the 24 B packed 3-bit numeric representation.')


PASS: scalar cases, highest bit, random distances/MNN, maximum distance 448.
Storage per descriptor: float32 256 B | uint8 ABS 64 B | packed thermometer 56 B.
Thermometer is 7 threshold planes, not the 24 B packed 3-bit numeric representation.


## 3. 외부 HPatches 쌍 구성

6개 viewpoint + 6개 illumination sequence에서 각각 1→2, 1→3: 총 24쌍. 리사이즈에 맞게 homography를 보정합니다. 경계 calibration에는 사용하지 않습니다. 이 평가는 리사이즈 좌표에서 3px 기준의 이미지 쌍 평가이며 공식 HPatches patch benchmark 점수는 아닙니다.


In [5]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h,
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/hpatches-sequences-release.zip
HPatches real-photo pairs: 24 (viewpoint and illumination)


## 4. C++ 커널 컴파일과 CPU 기능 확인

In [6]:
import ctypes, subprocess, tempfile, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
CPU_THREADS = 2
set_num_threads(CPU_THREADS)
torch.set_num_threads(CPU_THREADS)
faiss.omp_set_num_threads(CPU_THREADS)
cpp_source = '#include <cstdint>\n#include <vector>\n#include <algorithm>\n#include <immintrin.h>\n#include <omp.h>\nextern "C" int has_avx(){return __builtin_cpu_supports("avx512f") && __builtin_cpu_supports("avx512vpopcntdq");}\n__attribute__((target("popcnt"))) static inline int scalar(const uint64_t*a,const uint64_t*b){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[k]^b[k]);return d;}\n__attribute__((target("avx512f,avx512vpopcntdq"))) static int wide(const uint64_t*a,const uint64_t*b){\n __m512i x=_mm512_maskz_loadu_epi64(0x7f,a),y=_mm512_maskz_loadu_epi64(0x7f,b);\n return (int)_mm512_reduce_add_epi64(_mm512_popcnt_epi64(_mm512_xor_si512(x,y)));\n}\nextern "C" void encode_pack(const float*x,const float*e,int n,uint8_t*c,uint64_t*p){\n for(int i=0;i<n;i++){for(int k=0;k<7;k++)p[i*7+k]=0;\n for(int d=0;d<64;d++){int v=0;for(int k=0;k<7;k++)v+=x[i*64+d]>e[d*7+k];c[i*64+d]=v;\n for(int k=0;k<v;k++)p[i*7+k]|=uint64_t(1)<<d;}}\n}\nextern "C" void matrix(const uint64_t*a,const uint64_t*b,int n,int m,int threads,uint16_t*out){\n #pragma omp parallel for num_threads(threads) schedule(static)\n for(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=scalar(a+i*7,b+j*7);\n}\nextern "C" void fused(const uint64_t*a,const uint64_t*b,int n,int m,int mode,int threads,int*ab,int*ba){\n if(n==0||m==0)return;\n const bool avx=mode==1&&has_avx();\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\n #pragma omp parallel num_threads(threads)\n {\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n #pragma omp for schedule(static)\n for(int i=0;i<n;i++){int best=449,bj=m;\n for(int j=0;j<m;j++){int dist=0;\n if(mode==2){for(int k=0;k<7;k++){dist+=__builtin_popcountll(a[i*7+k]^b[j*7+k]);\n // Strict inequality preserves the smallest-index tie rule in both directions.\n if(dist>best&&dist>ds[j])break;}}\n else dist=avx?wide(a+i*7,b+j*7):scalar(a+i*7,b+j*7);\n if(dist<best){best=dist;bj=j;}\n if(dist<ds[j]||(dist==ds[j]&&i<ids[j])){ds[j]=dist;ids[j]=i;}\n }ab[i]=bj;}\n }\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){\n int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\n}\n'
compile_dir = Path(tempfile.mkdtemp(prefix='thermometer_hw_'))
(compile_dir/'kernel.cpp').write_text(cpp_source)
subprocess.run(['g++','-O3','-std=c++17','-shared','-fPIC','-fopenmp','-mpopcnt',
                str(compile_dir/'kernel.cpp'),'-o',str(compile_dir/'kernel.so')],check=True)
lib=ctypes.CDLL(str(compile_dir/'kernel.so'))
ptr=ctypes.c_void_p; integer=ctypes.c_int
lib.encode_pack.argtypes=[ptr,ptr,integer,ptr,ptr]
lib.matrix.argtypes=[ptr,ptr,integer,integer,integer,ptr]
lib.fused.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr,ptr]
lib.has_avx.restype=integer
for f in (lib.encode_pack,lib.matrix,lib.fused):f.restype=None
print('AVX-512 VPOPCNT available:',bool(lib.has_avx()), '| threads:',CPU_THREADS)
print(subprocess.run(['lscpu'],capture_output=True,text=True).stdout)

def adr(x):return x.ctypes.data_as(ptr)

def cpp_encode(x,edges):
    x=np.ascontiguousarray(x,dtype=np.float32);edges=np.ascontiguousarray(edges,dtype=np.float32)
    assert x.ndim==2 and x.shape[1]==64 and edges.shape==(64,7)
    codes=np.empty((len(x),64),np.uint8);packed=np.empty((len(x),7),np.uint64)
    lib.encode_pack(adr(x),adr(edges),len(x),adr(codes),adr(packed))
    return codes,packed

def cpp_matrix(a,b,threads=CPU_THREADS):
    a=np.ascontiguousarray(a,dtype=np.uint64);b=np.ascontiguousarray(b,dtype=np.uint64)
    out=np.empty((len(a),len(b)),np.uint16)
    lib.matrix(adr(a),adr(b),len(a),len(b),threads,adr(out));return out

def cpp_mnn(a,b,mode=0,threads=CPU_THREADS):
    a=np.ascontiguousarray(a,dtype=np.uint64);b=np.ascontiguousarray(b,dtype=np.uint64)
    if not len(a) or not len(b):return np.empty(0,np.int64),np.empty(0,np.int64)
    ab=np.empty(len(a),np.int32);ba=np.empty(len(b),np.int32)
    lib.fused(adr(a),adr(b),len(a),len(b),mode,threads,adr(ab),adr(ba))
    ii=np.arange(len(a));keep=ba[ab]==ii
    return ii[keep],ab[keep].astype(np.int64)

def binary_bytes(p):return np.ascontiguousarray(p).view(np.uint8).reshape(len(p),56)

def make_index(p):
    index=faiss.IndexBinaryFlat(448);index.add(binary_bytes(p));return index

def faiss_first(index,queries):
    q=binary_bytes(queries);d,ids=index.search(q,1);out=ids[:,0].copy()
    # Faiss k=1 ties need not follow NumPy's first-index rule.
    # Resolve all minimum-distance ties using an exact range search.
    for radius in np.unique(d[:,0]+1):
        rows=np.flatnonzero(d[:,0]+1==radius)
        lim,dist,labels=index.range_search(np.ascontiguousarray(q[rows]),int(radius))
        for t,row in enumerate(rows):out[row]=labels[lim[t]:lim[t+1]].min()
    return out

def faiss_mnn(a,b,index_a=None,index_b=None):
    if not len(a) or not len(b):return np.empty(0,np.int64),np.empty(0,np.int64)
    if index_a is None:index_a=make_index(a)
    if index_b is None:index_b=make_index(b)
    ab=faiss_first(index_b,a);ba=faiss_first(index_a,b)
    ii=np.arange(len(a));keep=ba[ab]==ii;return ii[keep],ab[keep]



AVX-512 VPOPCNT available: False | threads: 2
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           48 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               AuthenticAMD
Model name:                              AMD EPYC 7B12
CPU family:                              23
Model:                                   49
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4499.99
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ht syscall nx mmxext fxsr_opt pdpe1gb rdtscp lm constant_tsc rep_good nopl 

## 5. 커널 검증: empty·동점·최대 거리·경계 equality

In [7]:
# Finite inputs and ascending edges are required; no approximate search.
edge_np=np.ascontiguousarray(base_edges_cpu.numpy(),dtype=np.float32)
assert np.isfinite(edge_np).all() and (np.diff(edge_np,axis=1)>=0).all()
def cpu_encode_reference(x):
    return np.ascontiguousarray((x[:,:,None]>edge_np[None,:,:]).sum(axis=2),dtype=np.uint8)

def prepare_hw(p):
    x=np.ascontiguousarray(p['d1'].numpy(),dtype=np.float32)
    y=np.ascontiguousarray(p['d2'].numpy(),dtype=np.float32)
    assert np.isfinite(x).all() and np.isfinite(y).all()
    a,pa=cpp_encode(x,edge_np);b,pb=cpp_encode(y,edge_np)
    return dict(p,x=x,y=y,a=a,b=b,packed_a=pa,packed_b=pb,
                index_a=make_index(pa),index_b=make_index(pb))

MATCHERS={
 'float cosine':lambda p:mutual_from_scores((p['d1']@p['d2'].T).numpy(),True),
 'uint8 ABS matrix':lambda p:mutual_from_scores(pairwise_uint8_abs(p['a'],p['b']),False),
 'Numba thermometer matrix':lambda p:mutual_from_scores(pairwise_thermometer64(p['packed_a'],p['packed_b']),False),
 'C++ popcnt matrix':lambda p:mutual_from_scores(cpp_matrix(p['packed_a'],p['packed_b']),False),
 'C++ fused 1 thread':lambda p:cpp_mnn(p['packed_a'],p['packed_b'],0,1),
 'C++ fused 2 threads':lambda p:cpp_mnn(p['packed_a'],p['packed_b'],0,CPU_THREADS),
 'C++ fused exact pruning':lambda p:cpp_mnn(p['packed_a'],p['packed_b'],2,CPU_THREADS),
 'Faiss exact + first-index ties':lambda p:faiss_mnn(p['packed_a'],p['packed_b'],p['index_a'],p['index_b']),
}
if lib.has_avx():
 MATCHERS['C++ AVX512 fused']=lambda p:cpp_mnn(p['packed_a'],p['packed_b'],1,CPU_THREADS)
EXACT_COMPRESSED_METHODS=set(MATCHERS)-{'float cosine'}

def check_case(a,b):
    pa,pb=pack_thermometer64(a),pack_thermometer64(b)
    ref=pairwise_uint8_abs(a,b)
    np.testing.assert_array_equal(ref,cpp_matrix(pa,pb))
    expected=mutual_from_scores(ref,False)
    for mode in (0,1,2):
      for threads in (1,CPU_THREADS):
       got=cpp_mnn(pa,pb,mode,threads)
       for u,v in zip(expected,got):np.testing.assert_array_equal(u,v)
    for u,v in zip(expected,faiss_mnn(pa,pb)):np.testing.assert_array_equal(u,v)

test_rng=np.random.default_rng(SEED)
for n,m in ((1,1),(19,23),(31,7),(0,3),(3,0)):
 check_case(test_rng.integers(0,8,(n,64),dtype=np.uint8),test_rng.integers(0,8,(m,64),dtype=np.uint8))
check_case(np.zeros((9,64),np.uint8),np.zeros((11,64),np.uint8))
check_case(np.zeros((3,64),np.uint8),np.full((5,64),7,np.uint8))
z=np.vstack([edge_np[:,k] for k in range(7)]).astype(np.float32)
c,p=cpp_encode(z,edge_np)
np.testing.assert_array_equal(c,cpu_encode_reference(z))
np.testing.assert_array_equal(p,pack_thermometer64(c))
print('PASS: random/empty/tie/extreme cases and threshold equality.')



PASS: random/empty/tie/extreme cases and threshold equality.


## 6. 실제 이미지 정확도와 모든 exact MNN 검증

In [8]:
prepared_datasets={'synthetic':[prepare_hw(p) for p in validation_pairs]}
if real_pairs:prepared_datasets['HPatches']=[prepare_hw(p) for p in real_pairs]
accuracy_results=[]
for label,pairs in prepared_datasets.items():
 totals={name:dict(mutual=0,correct=0,errors=[]) for name in MATCHERS}
 for pair_idx,p in enumerate(pairs):
    np.testing.assert_array_equal(p['a'],cpu_encode_reference(p['x']))
    np.testing.assert_array_equal(p['b'],cpu_encode_reference(p['y']))
    ref_dist=pairwise_uint8_abs(p['a'],p['b'])
    np.testing.assert_array_equal(ref_dist,cpp_matrix(p['packed_a'],p['packed_b']))
    np.testing.assert_array_equal(ref_dist,pairwise_thermometer64(p['packed_a'],p['packed_b']))
    ref=mutual_from_scores(ref_dist,False)
    for name,fn in MATCHERS.items():
      i,j=fn(p)
      if name in EXACT_COMPRESSED_METHODS:
       for u,v in zip(ref,(i,j)):np.testing.assert_array_equal(u,v,err_msg=name)
      err=np.linalg.norm(project_points(p['k1'][i],p['H']).numpy()-p['k2'][j].numpy(),axis=1)
      t=totals[name];t['mutual']+=len(i);t['correct']+=int((err<GEOMETRIC_THRESHOLD).sum());t['errors']+=err.tolist()
 for name,t in totals.items():
    errors=t.pop('errors');accuracy_results.append(dict(dataset=label,method=name,**t,
      precision_pct=100*t['correct']/t['mutual'] if t['mutual'] else None,
      median_px=float(np.median(errors)) if errors else None,exact_abs_verified=name in EXACT_COMPRESSED_METHODS))
 print('Verified all exact methods:',label,len(pairs),'pairs')
display(pd.DataFrame(accuracy_results))



Verified all exact methods: synthetic 18 pairs
Verified all exact methods: HPatches 24 pairs


,dataset,method,mutual,correct,precision_pct,median_px,exact_abs_verified
0,synthetic,float cosine,8200,5611,68.426829,1.366410,False
1,synthetic,uint8 ABS matrix,8097,5433,67.098926,1.415135,True
2,synthetic,Numba thermometer matrix,8097,5433,67.098926,1.415135,True
3,synthetic,C++ popcnt matrix,8097,5433,67.098926,1.415135,True
4,synthetic,C++ fused 1 thread,8097,5433,67.098926,1.415135,True
5,synthetic,C++ fused 2 threads,8097,5433,67.098926,1.415135,True
6,synthetic,C++ fused exact pruning,8097,5433,67.098926,1.415135,True
7,synthetic,Faiss exact + first-index ties,8097,5433,67.098926,1.415135,True
8,HPatches,float cosine,10607,7138,67.295182,1.176445,False
9,HPatches,uint8 ABS matrix,10474,6921,66.077907,1.245561,True


## 7. SPEED: cached 및 인코딩 포함 총시간

In [9]:
def full_match(p,name):
    if name=='float cosine':return MATCHERS[name](p)
    if name in ('uint8 ABS matrix','Numba thermometer matrix'):
      a=cpu_encode_reference(p['x']);b=cpu_encode_reference(p['y'])
      if name=='uint8 ABS matrix':return mutual_from_scores(pairwise_uint8_abs(a,b),False)
      return mutual_from_scores(pairwise_thermometer64(pack_thermometer64(a),pack_thermometer64(b)),False)
    a,pa=cpp_encode(p['x'],edge_np);b,pb=cpp_encode(p['y'],edge_np)
    if name.startswith('Faiss'):return faiss_mnn(pa,pb) # BOTH index builds and tie resolution included
    pp=dict(p,a=a,b=b,packed_a=pa,packed_b=pb)
    return MATCHERS[name](pp)

def bench(fn):
    for _ in range(WARMUP):fn()
    samples=[]
    for _ in range(REPEATS):
      start=time.perf_counter_ns();fn();samples.append((time.perf_counter_ns()-start)/1e6)
    return dict(median_ms=float(np.median(samples)),p10_ms=float(np.percentile(samples,10)),
                p90_ms=float(np.percentile(samples,90)),samples_ms=samples)

timing_results=[];stage_results=[]
for label,pairs in prepared_datasets.items():
 for idx in np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int)):
    p=pairs[idx];names=list(MATCHERS);test_rng.shuffle(names)
    for name in names:
      cached=bench(lambda:MATCHERS[name](p));total=bench(lambda:full_match(p,name))
      timing_results.append(dict(dataset=label,pair=int(idx),method=name,n=len(p['a']),m=len(p['b']),
         cached_ms=cached['median_ms'],total_ms=total['median_ms'],p10_ms=total['p10_ms'],p90_ms=total['p90_ms'],
         cached_samples_ms=cached['samples_ms'],total_samples_ms=total['samples_ms']))
    stage_results.append(dict(dataset=label,pair=int(idx),
       reference_encode_pack_ms=bench(lambda:(pack_thermometer64(cpu_encode_reference(p['x'])),pack_thermometer64(cpu_encode_reference(p['y']))))['median_ms'],
       cpp_encode_pack_ms=bench(lambda:(cpp_encode(p['x'],edge_np),cpp_encode(p['y'],edge_np)))['median_ms'],
       numba_distance_ms=bench(lambda:pairwise_thermometer64(p['packed_a'],p['packed_b']))['median_ms'],
       cpp_distance_ms=bench(lambda:cpp_matrix(p['packed_a'],p['packed_b']))['median_ms']))
    print('Timed:',label,int(idx),len(p['a']),'x',len(p['b']))
frame=pd.DataFrame(timing_results)
summary=frame.groupby(['dataset','method'],sort=False)[['cached_ms','total_ms','p10_ms','p90_ms']].median().reset_index()
for label in summary.dataset.unique():
 mask=summary.dataset==label
 base=summary.loc[mask&(summary.method=='Numba thermometer matrix')].iloc[0]
 floating=summary.loc[mask&(summary.method=='float cosine')].iloc[0]
 summary.loc[mask,'cached_speedup_vs_thermo']=base.cached_ms/summary.loc[mask,'cached_ms']
 summary.loc[mask,'total_speedup_vs_thermo']=base.total_ms/summary.loc[mask,'total_ms']
 summary.loc[mask,'total_speedup_vs_float']=floating.total_ms/summary.loc[mask,'total_ms']
print('SPEED: median of per-pair medians; XFeat extraction excluded; total is directly measured.')
display(summary.round(4));display(pd.DataFrame(stage_results).round(4))
print('Faiss cached excludes index build; total includes both builds and deterministic tie resolution.')
print('Numba uses CPU quantization here; prior GPU encoding-inclusive timings are not directly comparable.')



Timed: synthetic 0 800 x 800
Timed: synthetic 4 800 x 800
Timed: synthetic 8 800 x 800
Timed: synthetic 12 800 x 800
Timed: synthetic 17 800 x 800
Timed: HPatches 0 800 x 800
Timed: HPatches 5 800 x 800
Timed: HPatches 11 800 x 800
Timed: HPatches 17 800 x 800
Timed: HPatches 23 800 x 800
SPEED: median of per-pair medians; XFeat extraction excluded; total is directly measured.


,dataset,method,cached_ms,total_ms,p10_ms,p90_ms,cached_speedup_vs_thermo,total_speedup_vs_thermo,total_speedup_vs_float
0,synthetic,C++ fused 2 threads,1.5577,3.1649,3.1354,3.2625,1.3560,2.1274,0.5299
1,synthetic,C++ fused 1 thread,1.6480,3.2933,3.2649,3.3424,1.2816,2.0445,0.5092
2,synthetic,Faiss exact + first-index ties,24.2856,26.4020,25.9144,27.2122,0.0870,0.2550,0.0635
3,synthetic,float cosine,1.7287,1.6770,1.6483,1.7363,1.2218,4.0150,1.0000
4,synthetic,C++ fused exact pruning,4.1446,5.6782,5.6361,6.0654,0.5096,1.1858,0.2953
5,synthetic,Numba thermometer matrix,2.1122,6.7331,6.6894,7.2417,1.0000,1.0000,0.2491
6,synthetic,uint8 ABS matrix,5.1716,9.6001,9.4827,10.2138,0.4084,0.7014,0.1747
7,synthetic,C++ popcnt matrix,2.0368,3.6546,3.6377,3.7530,1.0370,1.8424,0.4589
8,HPatches,uint8 ABS matrix,5.1795,9.5995,9.5433,10.3226,0.4069,0.7014,0.1779
9,HPatches,float cosine,1.6988,1.7074,1.6682,1.7555,1.2407,3.9434,1.0000


,dataset,pair,reference_encode_pack_ms,cpp_encode_pack_ms,numba_distance_ms,cpp_distance_ms
0,synthetic,0,4.4673,1.5847,1.2794,1.2982
1,synthetic,4,4.4711,1.5824,1.2790,1.3096
2,synthetic,8,4.5026,1.5762,1.2702,1.3006
3,synthetic,12,4.4834,1.5812,1.4410,1.3447
4,synthetic,17,5.9039,1.5955,1.2764,1.3179
5,HPatches,0,4.4848,1.5929,1.2746,1.3007
6,HPatches,5,4.4451,1.5781,1.2778,1.3007
7,HPatches,11,4.5033,1.5895,1.2689,1.2952
8,HPatches,17,4.4426,1.6129,1.2665,1.3533
9,HPatches,23,4.4667,1.5808,1.2697,1.2925


Faiss cached excludes index build; total includes both builds and deterministic tie resolution.
Numba uses CPU quantization here; prior GPU encoding-inclusive timings are not directly comparable.


## 8. 결과 저장

In [10]:
from datetime import datetime
from zoneinfo import ZoneInfo
run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S')
result_dir=data_dir/'experiment_results'/('thermometer_hardware_'+run_id)
result_dir.mkdir(parents=True,exist_ok=True)
summary.to_csv(result_dir/'speed_summary.csv',index=False)
pd.DataFrame(accuracy_results).to_csv(result_dir/'accuracy.csv',index=False)
report=dict(experiment='fixed quantile thermometer hardware acceleration',run_id=run_id,
 seed=SEED,top_k=TOP_K,threads=CPU_THREADS,warmup=WARMUP,repeats=REPEATS,
 avx512_vpopcnt_available=bool(lib.has_avx()),platform=platform.platform(),
 cpu_info=subprocess.run(['lscpu'],capture_output=True,text=True).stdout,
 versions=dict(numpy=np.__version__,torch=torch.__version__,faiss=getattr(faiss,'__version__','unknown')),
 accuracy=accuracy_results,timing=timing_results,stages=stage_results,
 note='Exact 8-bin ABS; first-index ties. CPU encoding included in total; XFeat and calibration excluded.')
(result_dir/'results.json').write_text(json.dumps(report,indent=2,allow_nan=False))
np.savez(result_dir/'fixed_edges.npz',edges=edge_np)
print('Results saved:',result_dir)


Results saved: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/experiment_results/thermometer_hardware_20261006_182110
